## 결정트리 분석

In [16]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# print("X_train:", X_train)
# print("X_test:", X_test)
# print("y_train:", y_train)
# print("y_test:", y_test)


In [9]:
def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return 1 - np.sum(p ** 2)

In [10]:
def best_split(X, y):
    n_features = X.shape[1]
    best = {"gini": float("inf"), "feature": None, "threshold": None}

    for feature in range(n_features):
        for threshold in np.unique(X[:, feature]):
            left = y[X[:, feature] <=threshold]
            right = y[X[:, feature] > threshold]
            if len(left) == 0 or len(right) == 0:
                continue
            w = (len(left) * gini(left) + len(right) * gini(right)) / len(y)
            if w < best["gini"]:
                best = {"gini": w, "feature": feature, "threshold": threshold}
    return best

In [11]:
def build_tree(X, y, depth=0, max_depth=3):
    if gini(y) == 0 or depth >= max_depth:
        return {"leaf": True, "label": np.bincount(y).argmax()}

    s = best_split(X, y)
    if s["feature"] is None:
        return {"leaf": True, "label": np.bincount(y).argmax()}

    mask = X[:, s["feature"]] <= s["threshold"]
    left = build_tree(X[mask], y[mask], depth + 1, max_depth)
    right = build_tree(X[~mask], y[~mask], depth + 1, max_depth)
    return {"leaf": False, "feature": s["feature"],
           "threshold": s["threshold"], "left": left, "right": right}

In [12]:
def predict_one(node, x):
    if node["leaf"]:
        return node["label"]
    if x[node["feature"]] <= node["threshold"]:
        return predict_one(node["left"], x)
    else:
        return predict_one(node["right"], x)

def predict(tree, X):
    return np.array([predict_one(tree, x) for x in X])

In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

my_tree = build_tree(X_train, y_train, max_depth=3)
my_pred = predict(my_tree, X_test)

sk = DecisionTreeClassifier(criterion="gini", max_depth=3, random_state=42)
sk.fit(X_train, y_train)
sk_pred = sk.predict(X_test)

print("내 구현 정확도 :", accuracy_score(y_test, my_pred))
print("sklearn 정확도 :", accuracy_score(y_test, sk_pred))

내 구현 정확도 : 0.9666666666666667
sklearn 정확도 : 0.9666666666666667
